# NewsBot Intelligence System | ITAI 2373 Midterm
**Student:** José Miguel Tuozzo Izaguirre  
**Project:** [GitHub repository](https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System)  
**Date:** October 2026

NewsBot combines eight NLP course modules into a historical-news analysis workflow. It uses BBC articles to study the language of five editorial topics, evaluates supervised article classification, and brings category, sentiment, grammatical patterns and named entities together in one inference pipeline.

**Audience and purpose:** Newsroom editors and media analysts could use a tool like this to suggest article routing and surface topics or names worth examining. Predictions remain subject to editorial judgment; the system does not verify factual claims.

**Reproducibility:** This notebook starts without saved outputs and can be executed using the labeled BBC News Train.csv. Dataset provenance, run metrics and the eight experiment figures are published in the [results folder](https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System/tree/main/outputs). An [archived, completed execution](https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System/actions/runs/38491913062) is available separately.

**Coverage:** business, entertainment, politics, sport and tech. Historical BBC text does not represent current news or a health category.

## 0. Setup, reproducibility, dependencies
Target **Google Colab Free**, CPU only. Run cells in order. On Colab, execute setup once; runtime might ask to restart after installs. The notebook uses a licensed/approved CSV uploaded by the team. It does **not** upload or redistribute the BBC articles through GitHub. No Kaggle API token is needed when the dataset is uploaded manually.

In [ ]:
%pip -q install "pandas>=2.0,<3" "numpy>=1.24,<3" "scikit-learn>=1.3,<2" "spacy>=3.7,<4" "nltk>=3.8,<4" "matplotlib>=3.7,<4" "seaborn>=0.12,<1" "joblib>=1.3,<2" "networkx>=3.0,<4"
import sys, subprocess, spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
import nltk
for item in ['vader_lexicon']:
    nltk.download(item, quiet=True)
print('Runtime environment ready. Use Colab Runtime > Restart if imports fail after upgrades.')

In [ ]:
import os, re, html, json, time, warnings, hashlib, platform
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from IPython.display import display
import spacy
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.base import clone
from scipy.special import softmax
from joblib import dump
SEED=42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='deep')
OUT=Path('outputs');OUT.mkdir(exist_ok=True, parents=True)
DATA=Path('data');DATA.mkdir(exist_ok=True, parents=True)
nlp=spacy.load('en_core_web_sm')
assert {'tagger','parser','ner','lemmatizer'}.issubset(nlp.pipe_names),('The full en_core_web_sm model is required for modules 4, 5 and 8: '+str(nlp.pipe_names))
nlp.max_length=2_000_000
sia=SentimentIntensityAnalyzer()
print({'python':platform.python_version(),'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,'spaCy':spacy.__version__})

## Module 1 — Data acquisition, provenance, cleaning and application context
**Selected source:** [Kaggle BBC News Classification](https://www.kaggle.com/competitions/learn-ai-bbc/data), **BBC News Train.csv** (1,490 labeled articles, fields `ArticleId`, `Text`, `Category`, five categories). The competition's unlabeled test file must **not** be used to compute evaluation accuracy. Our labeled train file is re-split into internal training and untouched holdout test subsets.

**Before use:** obtain written instructor approval. Download via Kaggle after signing in and accepting rules, then upload the CSV to Colab through Files. Alternatively manually set `DATA_PATH` to the file's path. The loader does not silently substitute ten artificial articles or another dataset.

**Dataset approval request** is in `docs/Dataset_Approval_Request.md`. Dataset contents and published articles remain the copyright of their owners. Do not redistribute them in the public GitHub repository.

In [ ]:
# Source: Kaggle learn-ai-bbc recommended by the assignment. Prefer your own Kaggle CSV.
# Fallback: educational research copy of the same 1,490-record labeled file hosted
# publicly by alvinrach/learn-ai-bbc on GitHub. Never commit the full article texts.
# BBC owns the article copyright; comply with source terms for any reuse.
# The fallback source and SHA-256 are logged for reproducible provenance.
DATA_PATH = 'BBC News Train.csv'
choices=[Path(DATA_PATH), DATA/'BBC News Train.csv',Path('/content/BBC News Train.csv')]
found=next((p for p in choices if p.is_file()),None)
DATA_ORIGIN='user_supplied_Kaggle_CSV' if found else 'public_research_mirror_of_Kaggle_competition_CSV'
if found is None:
    import urllib.request
    public_mirror='https://raw.githubusercontent.com/alvinrach/learn-ai-bbc/main/BBC%20News%20Train.csv'
    mirror_destination=DATA/'BBC News Train.csv'
    try:
        print('No local training CSV; retrieving the public research mirror. Source:',public_mirror)
        urllib.request.urlretrieve(public_mirror,mirror_destination)
        if mirror_destination.stat().st_size<100000:
            raise ValueError('Mirror file too small to be a valid BBC training corpus')
        found=mirror_destination
    except Exception as exc:
        raise FileNotFoundError(
            'Automatic retrieval failed. Download BBC News Train.csv from '
            'https://www.kaggle.com/competitions/learn-ai-bbc/data '
            'and upload to the Colab Files panel. Underlying error: '+str(exc)) from exc
raw=pd.read_csv(found,encoding='utf-8-sig')
cols={str(c).lower().strip().replace(' ',''):c for c in raw.columns}
# Kaggle copies appear with either Text or Article as the content header.
text_header=next((cols[k] for k in ['text','article','content'] if k in cols),None)
if 'articleid' not in cols or 'category' not in cols or text_header is None:
    raise ValueError(f'Expected ArticleId, Text/Article, Category. Received: {list(raw.columns)}. '
                     'You may have uploaded the unlabeled test file or the wrong dataset.')
df=raw.rename(columns={cols['articleid']:'article_id',text_header:'content',cols['category']:'category'})[['article_id','content','category']].copy()
initial=int(len(df))
missing=int(df[['content','category']].isna().any(axis=1).sum())
df=df.dropna(subset=['content','category']).copy()
df['content']=df.content.astype(str).str.strip()
df['category']=df.category.astype(str).str.lower().str.strip()
empty=int((df.content.eq('')|df.category.eq('')).sum())
df=df[(df.content.ne(''))&(df.category.ne(''))].copy()
# EXACT duplicates are removed before splitting, avoiding duplicate documents in training/test.
duplicates=int(df.duplicated(subset='content',keep='first').sum())
df=df.drop_duplicates(subset='content',keep='first').reset_index(drop=True)
assert df.article_id.notna().all() and df.article_id.is_unique,'Missing or repeated ArticleIds.'
assert pd.to_numeric(df.article_id,errors='coerce').notna().all(),'ArticleId must be numeric.'
df.article_id=df.article_id.astype(int)
df['full_text']=df.content
# The historical BBC train CSV usually includes the headline followed by the article body.
df['title']=df.content.str.slice(0,85)
df['word_count']=df.full_text.str.split().str.len()
allowed={'business','entertainment','politics','sport','tech'}
assert set(df.category).issubset(allowed),f'Unexpected BBC labels: {set(df.category)}'
assert 500<=len(df)<=2000 and df.category.nunique()>=4,'Dataset outside instructor limits'
assert df.category.value_counts().min()>=5,'Not enough instances for stratified 5-fold CV.'
assert (df.word_count>=50).mean()>=.98,'Most documents must contain at least 50 words.'
assert df[['content','category']].notna().all().all()
print('Input file:',str(found.resolve()),'| source:',DATA_ORIGIN)
print('Original rows:',initial,'| Missing rows:',missing,'| Empty rows:',empty,
      '| Removed exact duplicate article texts:',duplicates,'| Retained:',len(df))
print('Classes:',df.category.value_counts().to_dict())
print('Number of words:',df.word_count.describe().round(1).to_dict())
print('Data provenance: historical BBC, Kaggle competition; not live or representative of 2026 news.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
sns.countplot(data=df,y='category',order=df.category.value_counts().index,color='#3186a8',ax=axs[0]);axs[0].set_title('Articles by topic')
sns.boxplot(data=df,x='word_count',y='category',ax=axs[1]);axs[1].set_title('Article length by topic');axs[1].set_xlim(left=0)
plt.tight_layout();plt.savefig(OUT/'01_data_distribution.png',dpi=160,bbox_inches='tight');plt.show()
print('Sampling check:',df.groupby('category').word_count.agg(['count','median','mean']).round(1).to_string())

## Module 2 — Preprocessing and controlled edge cases
Three views are kept separate: (1) **raw text** for spaCy NER and parsing, (2) **lightly normalized classification text** preserving negation, money, years and punctuation-related meaning, and (3) **lemmatized lexical view** for linguistic analysis. This avoids the common error of deleting `not`, names and numerical facts before NER/sentiment. We do not remove BBC text by substring or accidentally leak the label into the features.

In [ ]:
def normalize_text(value):
    if value is None or pd.isna(value): return ''
    text=html.unescape(str(value))
    text=re.sub(r'<[^>]*>',' ',text)
    text=re.sub(r'https?://\S+|www\.\S+',' URL ',text,flags=re.I)
    text=re.sub(r'(?<!\S)[\w.\-+]+@[\w.\-]+\.[a-zA-Z]{2,}',' EMAIL ',text)
    return re.sub(r'\s+',' ',text).strip()

def lexical_tokens(text):
    doc=nlp.make_doc(normalize_text(text))
    return [t.lower_ for t in doc if t.is_alpha and not t.is_stop and len(t)>=3]

def lemmatized_text(text):
    # True spaCy lemmatizer, without stripping original input used by syntax/NER
    doc=nlp(normalize_text(text)[:2500])
    return ' '.join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and len(t)>=3)

cases=['<p>Apple &amp; Microsoft</p>','Not a good result: $2,000 in 2025!',
       'Contact abc@example.com at https://example.com','  multi  space  ',None,'Café & London.']
for t in cases:print(repr(t),'->',repr(normalize_text(t)))
assert normalize_text(None)==''
assert 'Not' in normalize_text(cases[1])
assert '2025' in normalize_text(cases[1])
assert 'example.com' not in normalize_text(cases[2])
df['text_clean']=df.full_text.map(normalize_text)
df['tokens_lexical']=df.text_clean.map(lexical_tokens)
print('Processed:',len(df),'docs, empty after clean:',(df.text_clean.str.len()==0).sum())

## Module 3 — TF-IDF, discriminative terms and feature choices
**Important methodology:** descriptive full-corpus TF-IDF below is for **exploratory category profiles only**. All predictive TF-IDF weights and hyperparameters are fit **inside training folds** later, never on the test set. Sparse matrices are preserved; converting 8,000+ features to a dense DataFrame is intentionally avoided.

**Why descriptive TF-IDF is different from predictive TF-IDF:** The exploratory terms below summarize all articles and are never fed as precomputed vectors to the supervised model. Predictive TF-IDF is fitted afresh in each training fold. This distinction prevents data leakage.

In [ ]:
explore_vec=TfidfVectorizer(max_features=6000,min_df=3,max_df=.9,ngram_range=(1,2),sublinear_tf=True,stop_words='english')
explore_X=explore_vec.fit_transform(df.text_clean)
terms=np.array(explore_vec.get_feature_names_out())
term_profiles={}
for cat in sorted(df.category.unique()):
    mask=(df.category.to_numpy()==cat)
    # Mean category TF-IDF minus mean outside category isolates comparatively distinctive terms.
    score=np.asarray(explore_X[mask].mean(axis=0)).ravel()-np.asarray(explore_X[~mask].mean(axis=0)).ravel()
    ids=np.argsort(score)[-12:][::-1]
    term_profiles[cat]=[(str(terms[k]),round(float(score[k]),4)) for k in ids]
    print(cat,':',', '.join(t for t,_ in term_profiles[cat][:10]))
fig,axs=plt.subplots(len(term_profiles),1,figsize=(11,13))
for ax,(cat,features) in zip(axs,term_profiles.items()):
    features=features[:8][::-1]
    ax.barh([x[0] for x in features],[x[1] for x in features]);ax.set_title(cat.capitalize()+' distinctive TF-IDF terms')
plt.tight_layout();plt.savefig(OUT/'02_tfidf_category_terms.png',dpi=160,bbox_inches='tight');plt.show()
print('Descriptive vector vocabulary:',explore_X.shape[1], 'sparsity:',round(1-explore_X.nnz/(explore_X.shape[0]*explore_X.shape[1]),5))

## Module 4 — Part-of-speech (POS) patterns
Use spaCy's grammatical tags on **original articles** (preserving capitalized proper nouns); proportions use **non-space, non-punctuation tokens** as their denominator. For runtime efficiency, the shared spaCy batch in Module 5 supplies both POS and parse/NER information.

## Module 5 — Syntax and semantics
The parser extracts dependency structures and approximate **subject–verb–object (SVO)** relations. These are **dependency-based heuristics**, not gold semantic-role labels: passive voice, coordination and implied subjects can produce incomplete or misleading triples. We show this limitation explicitly.

In [ ]:
def analyze_pos_patterns(doc):
    t=[w for w in doc if not w.is_space and not w.is_punct]
    counts=Counter(w.pos_ for w in t)
    return {p:counts[p]/max(len(t),1) for p in ['NOUN','PROPN','VERB','ADJ','ADV','ADP','NUM','PRON','AUX']}

def extract_syntactic_features(doc):
    sub=['nsubj','nsubjpass','csubj','csubjpass']
    obj=['dobj','obj','iobj','attr','oprd']
    triples=[]
    for v in doc:
        if v.pos_ not in ('VERB','AUX'):continue
        left=[x for x in v.children if x.dep_ in sub]
        right=[x for x in v.children if x.dep_ in obj]
        for a in left[:2]:
            for b in right[:2]:
                triples.append((a.text,v.lemma_,b.text))
    dep_counts=Counter(t.dep_ for t in doc if not t.is_punct and not t.is_space)
    return {'n_sentences':sum(1 for _ in doc.sents),
            'n_noun_chunks':sum(1 for _ in doc.noun_chunks),
            'n_subjects':sum(t.dep_ in sub for t in doc),
            'n_objects':sum(t.dep_ in obj for t in doc),
            'n_passive':sum(t.dep_ in ('nsubjpass','auxpass') for t in doc),
            'n_named_entities':len(doc.ents),
            'svos':triples[:8], 'dep_counts':dict(dep_counts)}

sample=nlp('Microsoft acquired a startup, and analysts praised the agreement.')
print('POS:',analyze_pos_patterns(sample))
print('Syntax example:',extract_syntactic_features(sample))
print('Dependency edges:',[(x.text,x.dep_,x.head.text) for x in sample])

## Module 6 — Sentiment and emotional tone
Use VADER `compound` with thresholds ±0.05 for **lexical polarity**, not factual truth or political slant. The additional **emotion proxy** counts terms from a **small explicit dictionary**; it is an illustrative lexicon, **not** a validated six-emotion classifier. The dictionary and its major limitations are transparent.

In [ ]:
EMOTION_WORDS={
 'joy':{'celebrate','win','success','happy','delight','joy','victory','excite','award','praise'},
 'fear':{'fear','threat','risk','danger','worry','afraid','uncertain','crisis','alarm'},
 'anger':{'anger','angry','outrage','protest','furious','criticize','condemn','rage'},
 'sadness':{'sad','death','loss','grief','mourn','tragic','suffer','disaster'},
 'surprise':{'surprise','unexpected','shock','astonish','sudden','unusual'},
}
def analyze_sentiment(text):
    s=sia.polarity_scores(normalize_text(text))
    label='positive' if s['compound']>=.05 else 'negative' if s['compound']<=-.05 else 'neutral'
    return {**s,'sentiment_label':label}
def emotion_proxy(doc):
    tokens=[w.lemma_.lower() for w in doc if w.is_alpha]
    counts={k:sum(w in lex for w in tokens) for k,lex in EMOTION_WORDS.items()}
    total=max(sum(counts.values()),1)
    return {k:float(v/total) for k,v in counts.items()}
print('Sentiment:',analyze_sentiment('The team did not win the final match.'))
print('Emotion proxy:',emotion_proxy(nlp('A surprising victory led to joyful celebrations.')))

## Module 8 — Named entity recognition and co-mentions
This analysis is grouped with Modules 4–6 so the notebook can reuse the same spaCy parsing pass before training the classifier in Module 7.

NewsBot extracts `PERSON`, `ORG`, `GPE`, `DATE` and `MONEY` mentions from the original-case text. It also counts entities mentioned in the same article. These counts are exploratory: names may be recognized incorrectly, and a co-mention does not establish a real-world connection.

In [ ]:
KEEP_ENTS={'PERSON','ORG','GPE','DATE','MONEY'}
def extract_entities(doc):
    return [{'text':e.text,'label':e.label_,'start':e.start_char,'end':e.end_char}
            for e in doc.ents if e.label_ in KEEP_ENTS]

# Single shared batched pass for Modules 4, 5, 6, 8; parses up to 3,500 chars/article to bound runtime.
# Important: text truncation is only for linguistic statistics, NOT the supervised classification corpus.
LINGUISTIC_CHAR_LIMIT=3500
pos_rows=[];syn_rows=[];sent_rows=[];ent_rows=[];co_mentions=Counter();lemma_rows=[]
start=time.perf_counter()
texts=(str(x)[:LINGUISTIC_CHAR_LIMIT] for x in df.full_text)
for i,doc in enumerate(nlp.pipe(texts,batch_size=32)):
    cat=str(df.at[i,'category']); id_=int(df.at[i,'article_id'])
    lemma_rows.append(' '.join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and len(t)>=3))
    pos_rows.append({'article_id':id_,'category':cat,**analyze_pos_patterns(doc)})
    syn=extract_syntactic_features(doc)
    syn_rows.append({'article_id':id_,'category':cat,**{k:v for k,v in syn.items() if k not in ('svos','dep_counts')}, 'example_svo':syn['svos'][0] if syn['svos'] else None})
    ss=analyze_sentiment(df.at[i,'full_text'])  # full original article
    sent_rows.append({'article_id':id_,'category':cat,**ss,**{'emotion_'+k:v for k,v in emotion_proxy(doc).items()}})
    ents=extract_entities(doc)
    for e in ents:ent_rows.append({'article_id':id_,'category':cat,**e})
    # prevent one name repeated in article from overweighting co-mention links
    unique=sorted({e['text'].lower().strip() for e in ents if e['label'] in ('PERSON','ORG') and len(e['text'])>=4})[:50]
    co_mentions.update(combinations(unique,2))
    if (i+1)%300==0:print('NLP docs:',i+1)
df['text_lemma']=lemma_rows
pos_df=pd.DataFrame(pos_rows); syntax_df=pd.DataFrame(syn_rows)
sentiment_df=pd.DataFrame(sent_rows); entities_df=pd.DataFrame(ent_rows)
print('Parsed',len(pos_df),'articles in',round(time.perf_counter()-start,1),'s. Entities:',len(entities_df))
assert len(pos_df)==len(df)==len(syntax_df)==len(sentiment_df)
assert not entities_df.empty,'NER unexpectedly produced no entities'
print('Examples of lemmatization:',df[['category','text_lemma']].head(2).to_dict('records'))
print('Examples of approximate SVO triples:',syntax_df.example_svo.dropna().head(8).tolist())

In [ ]:
# POS category comparisons
pos_cols=['NOUN','PROPN','VERB','ADJ','ADV','ADP','NUM','PRON','AUX']
fig,axs=plt.subplots(2,2,figsize=(14,10))
sns.heatmap(pos_df.groupby('category')[pos_cols].mean(),annot=True,fmt='.2f',cmap='Blues',ax=axs[0,0]);axs[0,0].set_title('POS proportions (truncated original text)')
sns.barplot(data=sentiment_df,x='category',y='compound',errorbar=('ci',95),color='#3186a8',ax=axs[0,1]);axs[0,1].tick_params(axis='x',rotation=30);axs[0,1].set_title('Mean VADER compound and 95% CI')
sentiment_df.groupby(['category','sentiment_label']).size().unstack(fill_value=0).pipe(lambda t:t.div(t.sum(axis=1),axis=0)).plot.bar(stacked=True,ax=axs[1,0]);axs[1,0].set_title('Share of sentiment labels');axs[1,0].legend(fontsize=8)
sns.barplot(data=syntax_df,x='category',y='n_subjects',errorbar=('ci',95),color='#3186a8',ax=axs[1,1]);axs[1,1].tick_params(axis='x',rotation=30);axs[1,1].set_title('Dependency subjects per truncated article')
plt.tight_layout();plt.savefig(OUT/'03_linguistic_sentiment.png',dpi=160,bbox_inches='tight');plt.show()
print('Sentiment by category:');display(sentiment_df.groupby('category').compound.agg(['mean','median','std','count']).round(3))
print('Proxy emotion scores (NOT validated emotion labels):');display(sentiment_df.groupby('category')[[c for c in sentiment_df if c.startswith('emotion_')]].mean().round(3))
print('Syntax patterns:');display(syntax_df.groupby('category')[['n_sentences','n_noun_chunks','n_subjects','n_objects','n_passive']].mean().round(2))

In [ ]:
entity_type_order=['PERSON','ORG','GPE','DATE','MONEY']
entity_counts=pd.crosstab(entities_df.category,entities_df.label).reindex(columns=entity_type_order,fill_value=0)
entity_per_doc=entity_counts.div(df.category.value_counts().reindex(entity_counts.index),axis=0)
fig,axs=plt.subplots(1,2,figsize=(14,4))
sns.heatmap(entity_per_doc,annot=True,fmt='.1f',cmap='YlGnBu',ax=axs[0]);axs[0].set_title('NER mentions per article (first 3,500 chars)')
entities_df[entities_df.label.isin(['ORG','PERSON'])].text.str.lower().value_counts().head(12).sort_values().plot.barh(ax=axs[1]);axs[1].set_title('Most mentioned named entities')
plt.tight_layout();plt.savefig(OUT/'04_ner_analysis.png',dpi=160,bbox_inches='tight');plt.show()
print('Top PERSON/ORG co-mentions (not proven relationships):',co_mentions.most_common(12))
# Co-mention visual, top edge count; this is NOT a verified semantic graph.
graph=nx.Graph()
for (a,b),w in co_mentions.most_common(15):
    graph.add_edge(a,b,weight=w)
fig,ax=plt.subplots(figsize=(11,6))
if graph.number_of_edges():
    layout=nx.spring_layout(graph,seed=SEED)
    widths=[0.5+2*np.log1p(graph[u][v]['weight']) for u,v in graph.edges()]
    nx.draw_networkx(graph,pos=layout,ax=ax,with_labels=True,node_size=750,font_size=7,
        node_color='#b4dce9',edge_color='#64748b',width=widths)
    ax.set_title('Top PERSON/ORG same-article co-mentions (NOT factual links)')
else:
    ax.text(.5,.5,'No same-article PERSON/ORG co-mentions in the analyzed sample',ha='center',va='center',transform=ax.transAxes)
ax.axis('off');plt.tight_layout();plt.savefig(OUT/'07_entity_comentions.png',dpi=160,bbox_inches='tight');plt.show()
print('Entity-by-category table:');display(entity_counts)
print('Top by entity label:')
for typ in entity_type_order:
 print(typ,entities_df.loc[entities_df.label==typ,'text'].str.lower().value_counts().head(5).to_dict())

## Module 7 — Multi-class classification and model evaluation
The model comparison uses a **stratified 80/20 split**, with **5-fold stratified cross-validation only on training articles**. TF-IDF is inside each estimator pipeline, preventing vocabulary/IDF leakage. The **selection criterion is training-fold macro-F1**, which gives minority categories equal influence. The holdout stays untouched until after model choice. No hyperparameters are chosen by inspecting test performance. We also check exact duplicate text *before* splitting.

Models: Dummy most-frequent baseline, Multinomial Naive Bayes, Logistic Regression, Linear SVM. Candidates tune TF-IDF n-gram feature choice within training-only grid. `LinearSVC` margins are **not calibrated probabilities** and are never mislabeled as probabilities. Confidence shown at inference is either logistic `predict_proba` or a normalized *relative margin score* clearly labeled as uncalibrated.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(
 df.text_clean.astype(str),df.category.astype(str),test_size=.2,random_state=SEED,stratify=df.category)
print('train/test',len(X_train),len(X_test));display(pd.concat([y_train.value_counts().rename('train'),y_test.value_counts().rename('test')],axis=1))
assert set(X_train.index).isdisjoint(X_test.index)
assert not set(X_train).intersection(set(X_test)),'Identical text leaked across splits'

def vectorizer(**extra):
    return TfidfVectorizer(strip_accents='unicode',lowercase=True,stop_words=None,
       sublinear_tf=True,max_df=.95,min_df=2,max_features=10000,ngram_range=(1,2),**extra)

models={
 'Dummy majority':DummyClassifier(strategy='most_frequent'),
 'Naive Bayes':MultinomialNB(alpha=.5),
 'Logistic Regression':LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED),
 'Linear SVM':LinearSVC(C=1.0,class_weight='balanced',random_state=SEED)
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
cv_rows={}
for name,model in models.items():
    pipeline=Pipeline([('tfidf',vectorizer()),('classifier',model)])
    score=cross_validate(pipeline,X_train,y_train,cv=cv,scoring={'macro_f1':'f1_macro','accuracy':'accuracy'},n_jobs=1,error_score='raise')
    cv_rows[name]={'name':name,'mean_cv_macro_f1':float(score['test_macro_f1'].mean()),
      'std_cv_macro_f1':float(score['test_macro_f1'].std()),'mean_cv_accuracy':float(score['test_accuracy'].mean())}
cv_df=pd.DataFrame(cv_rows.values()).sort_values('mean_cv_macro_f1',ascending=False).reset_index(drop=True)
display(cv_df.round(4));cv_df.to_csv(OUT/'model_cv_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4));sns.barplot(data=cv_df,y='name',x='mean_cv_macro_f1',color='#3186a8',ax=ax)
ax.set_xlim(0,1);ax.set_title('Training-set 5-fold CV macro-F1 (model selection)')
plt.tight_layout();plt.savefig(OUT/'05_model_cv_comparison.png',dpi=160,bbox_inches='tight');plt.show()

In [ ]:
# Feature-selection + hyperparameter optimization on TRAIN ONLY for the leading nontrivial model.
best_model_name=next(name for name in cv_df.name if name!='Dummy majority')
base_classifier=clone(models[best_model_name]);print('Best base family by train CV:',best_model_name)
param_grid={'tfidf__ngram_range':[(1,1),(1,2)],'tfidf__max_features':[5000,10000]}
if best_model_name=='Naive Bayes':param_grid['classifier__alpha']=[.3,1.0]
elif best_model_name=='Logistic Regression':param_grid['classifier__C']=[.5,2.0]
elif best_model_name=='Linear SVM':param_grid['classifier__C']=[.5,2.0]
search=GridSearchCV(Pipeline([('tfidf',vectorizer()),('classifier',base_classifier)]),param_grid,
   scoring='f1_macro',cv=cv,n_jobs=1,refit=True,error_score='raise')
search.fit(X_train,y_train)
best_model=search.best_estimator_
print('Tuned TRAIN-only mean CV macro-F1:',round(search.best_score_,4));print('Parameters:',search.best_params_)
# Single, untouched holdout evaluation AFTER final model selection.
pred=best_model.predict(X_test)
metrics={'accuracy':float(accuracy_score(y_test,pred)),
         'macro_f1':float(f1_score(y_test,pred,average='macro')),
         'weighted_f1':float(f1_score(y_test,pred,average='weighted'))}
print('HOLDOUT TEST:',metrics)
print(classification_report(y_test,pred,digits=3,zero_division=0))
fig,ax=plt.subplots(figsize=(8,6));ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=sorted(y_test.unique()),
 cmap='Blues',xticks_rotation=35,ax=ax,colorbar=False);ax.set_title('Untouched holdout confusion matrix')
plt.tight_layout();plt.savefig(OUT/'06_confusion_matrix.png',dpi=160,bbox_inches='tight');plt.show()
errors=pd.DataFrame({'text':X_test.values,'true':y_test.values,'predicted':pred})
errors=errors.loc[errors.true!=errors.predicted].copy()
errors['excerpt']=errors.text.str.slice(0,180)
print('Incorrect holdout predictions:',len(errors));display(errors[['true','predicted','excerpt']].head(8))
errors[['true','predicted','excerpt']].to_csv(OUT/'error_analysis_excerpt.csv',index=False)
(OUT/'classification_report.txt').write_text(classification_report(y_test,pred,digits=4,zero_division=0),encoding='utf-8')
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

In [ ]:
# Class-wise metrics and category-specific feature interpretation (model-dependent coefficients).
from sklearn.metrics import precision_recall_fscore_support
labels=sorted(y_test.unique())
p,r,f,s=precision_recall_fscore_support(y_test,pred,labels=labels,zero_division=0)
per_class=pd.DataFrame({'category':labels,'precision':p,'recall':r,'f1':f,'support':s})
display(per_class.round(3));per_class.to_csv(OUT/'per_class_metrics.csv',index=False)
clf=best_model.named_steps['classifier']; vocab=best_model.named_steps['tfidf'].get_feature_names_out()
if hasattr(clf,'coef_'):
    for i,cat in enumerate(clf.classes_):
        ix=np.argsort(clf.coef_[i])[-8:][::-1]
        print(cat,'top positive linear features:',list(vocab[ix]))
else:
    for i,cat in enumerate(clf.classes_):
        ix=np.argsort(clf.feature_log_prob_[i])[-8:][::-1]
        print(cat,'top NB lexical features:',list(vocab[ix]))

# TF-IDF ablation: compare unigrams only vs unigrams+bigrams on training data.
ablation_rows=[]
for grams in [(1,1),(1,2)]:
    ablation=clone(best_model).set_params(tfidf__ngram_range=grams)
    scores=cross_validate(ablation,X_train,y_train,cv=cv,scoring='f1_macro',n_jobs=1)
    score=float(scores['test_score'].mean())
    ablation_rows.append({'ngram_min':grams[0],'ngram_max':grams[1],'cv_macro_f1':score})
    print('Ablation ngrams=',grams,'train-CV macro-F1',round(score,4))
pd.DataFrame(ablation_rows).to_csv(OUT/'feature_ablation.csv',index=False)

## Research extension — Frozen-model robustness and sampling uncertainty
This stress test perturbs the held-out articles **after** model choice; it is a labeled sensitivity check, not unseen-publisher evaluation. The bootstrap interval is a within-corpus uncertainty summary, not a guarantee of deployment quality.


In [ ]:
# Advanced, deliberately bounded robustness test; NOT an external-dataset validation.
# The final model is frozen. Test the untouched holdout predictions with deterministic
# light character corruption, with no model selection based on this stress test.
def deterministic_light_typos(text,seed):
    rng=np.random.default_rng(seed)
    words=str(text).split()
    eligible=[j for j,w in enumerate(words) if len(w)>7 and w.isalpha()]
    for j in rng.choice(eligible,size=min(len(eligible),max(1,int(len(eligible)*.06))),replace=False):
        token=words[j]
        k=int(rng.integers(1,len(token)-1))
        words[j]=token[:k]+token[k+1]+token[k]+token[k+2:]
    return ' '.join(words)

perturbed=[deterministic_light_typos(t,SEED+j) for j,t in enumerate(X_test)]
robust_predictions=best_model.predict(perturbed)
robustness=pd.DataFrame([
 {'condition':'Original holdout','macro_f1':metrics['macro_f1'],'accuracy':metrics['accuracy']},
 {'condition':'6% eligible-token character swaps','macro_f1':float(f1_score(y_test,robust_predictions,average='macro')),
  'accuracy':float(accuracy_score(y_test,robust_predictions))}])
robustness.to_csv(OUT/'robustness_stress_test.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4))
ax.barh(robustness.condition,robustness.macro_f1,color=['#3186a8','#e09f5a'])
ax.set_xlim(0,1);ax.set_xlabel('Macro-F1');ax.set_title('Frozen-model stress test (synthetic typos, not temporal shift)')
plt.tight_layout();plt.savefig(OUT/'08_robustness_stress_test.png',dpi=160,bbox_inches='tight');plt.show()
print('Robustness stress results:');display(robustness.round(4))
# Bootstrap shows sampling uncertainty within this historical test sample;
# does not address domain shift or systematic source bias.
bootstrap_rng=np.random.default_rng(SEED)
truth=np.asarray(y_test); guesses=np.asarray(pred)
bootstrap_scores=[]
for _ in range(800):
    ix=bootstrap_rng.integers(0,len(truth),len(truth))
    bootstrap_scores.append(float(f1_score(truth[ix],guesses[ix],average='macro',labels=sorted(np.unique(truth)),zero_division=0)))
macro_f1_ci=[round(float(x),4) for x in np.percentile(bootstrap_scores,[2.5,97.5])]
(OUT/'bootstrap_uncertainty.json').write_text(json.dumps({'holdout_macro_f1_95pct_bootstrap_interval':macro_f1_ci,
 'bootstrap_resamples':800,'limitations':'within-sample sampling uncertainty only; not external or time-shift validation'},indent=2))
print('Bootstrap 95% interval for holdout macro-F1:',macro_f1_ci)


## Integrated module outputs and business interpretation
A high macro-F1 on historical within-corpus articles may support *assisted* desk routing; it does not justify unsupervised publishing or claims about fresh news. Review all cross-topic articles and abstain on low-confidence predictions. The POS patterns and NER values below are exploratory, not causal. Categories are compared within the same sampling frame; corpus makeup and old dates dominate interpretation.

In [ ]:
insights={
 'dataset':{'articles':int(len(df)),'source':'Kaggle BBC News Train.csv','labels':df.category.value_counts().to_dict(),
            'missing_removed':missing,'exact_text_duplicates_removed':duplicates,'dataset_status':'INSTRUCTOR_APPROVAL_REQUIRED'},
 'evaluation':{'selected_algorithm':best_model_name,'cross_validation_macro_f1':round(float(search.best_score_),4),
               'holdout':metrics,'holdout_articles':int(len(y_test))},
 'sentiment':{'mean_vader_by_category':sentiment_df.groupby('category').compound.mean().round(4).to_dict()},
 'distinctive_tfidf_terms':{k:[{'term':t,'weight_difference':v} for t,v in row[:8]] for k,row in term_profiles.items()},
 'class_performance':per_class.round(4).to_dict('records'),
 'cross_validation_models':cv_df.round(4).to_dict('records'),
 'misclassified_holdout_count':int(len(errors)),
 'feature_ablation':ablation_rows,
 'robustness_stress_test':robustness.to_dict('records'),
 'bootstrap_macro_f1_95pct_ci':macro_f1_ci,
 'frequent_confusions':[{'actual':a,'predicted':b,'count':int(count)} for (a,b),count in Counter(zip(errors.true,errors.predicted)).most_common(5)],
 'ner':{'mentions_by_type':entities_df.label.value_counts().to_dict(),
        'top_comentions':[[a,b,int(v)] for (a,b),v in co_mentions.most_common(15)]},
 'pos':{c:pos_df.groupby('category')[c].mean().round(4).to_dict() for c in pos_cols},
 'syntax':{'mean_passives_by_category':syntax_df.groupby('category').n_passive.mean().round(3).to_dict()},
 'limitations':['Historical 2004–2005 single-source BBC coverage; do not infer 2026 trends',
 'Dataset has 5 categories, not health','Sentiment lexicon is not an objective fact checker',
 'Emotion proxy is an illustrative dictionary, not a trained emotion classifier',
 'Entity co-mentions do not imply real-world relationships','NER/parse statistics truncate after 3,500 characters',
 'Cross-validation and heldout testing are in-domain; generalization to recent stories is unproven']
}
(OUT/'insights.json').write_text(json.dumps(insights,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(insights['evaluation'],indent=2))
print('Managerial recommendations: (1) human-in-the-loop routing; (2) flag low-margin cases;',
      '(3) periodic new-article validation; (4) manual review of entity matches;',
      '(5) never equate polarity with factual accuracy.')

In [ ]:
# AUTOMATIC EVIDENCE-BASED DISCUSSION: computed from the executed BBC results.
print('\nBUSINESS INTERPRETATION (verify against individual articles):')
best=per_class.sort_values('f1',ascending=False).iloc[0]
weakest=per_class.sort_values('f1').iloc[0]
print(f'1. Model reliability: holdout macro-F1 {metrics["macro_f1"]:.3f}; '
      f'best class {best.category} (F1 {best.f1:.3f}); '
      f'weakest class {weakest.category} (F1 {weakest.f1:.3f}). '
      'The weak category is a priority for human review and targeted error checks.')
sent_means=sentiment_df.groupby('category').compound.mean().sort_values()
print(f'2. Article tone: lowest mean VADER category {sent_means.index[0]} '
      f'({sent_means.iloc[0]:.3f}), highest {sent_means.index[-1]} '
      f'({sent_means.iloc[-1]:.3f}). Lexical tone does not establish author stance.')
if not entities_df.empty:
    top_ent=entities_df.loc[entities_df.label.isin(['PERSON','ORG']),'text'].str.lower().value_counts().head(1)
    if len(top_ent):
        print(f'3. Entity monitoring: most frequent PERSON/ORG mention {top_ent.index[0]} '
              f'({int(top_ent.iloc[0])} occurrences in analyzed text). Verify model tags before acting.')
if len(errors):
    print('4. Holdout disagreement example (excerpt only, not a verified news fact):')
    display(errors[['true','predicted','excerpt']].head(1))
else:
    print('4. No errors observed on this holdout. This does NOT imply perfect generalization.')
from pathlib import Path
summary=(f'Actual labeled sample: {len(df)} articles.\n'
         f'Selected model: {best_model_name}. Training-only tuned CV macro-F1: {search.best_score_:.4f}.\n'
         f'Untouched holdout accuracy: {metrics["accuracy"]:.4f}; macro-F1: {metrics["macro_f1"]:.4f}.\n'
         f'Weakest holdout class: {weakest.category}, F1={weakest.f1:.4f}.\n'
         f'Highest mean VADER category: {sent_means.index[-1]}, {sent_means.iloc[-1]:.4f}.\n'
         f'Entity co-mentions are associative, NOT semantic edges.\n')
(OUT/'EVIDENCE_BASED_FINDINGS.txt').write_text(summary,encoding='utf-8')
print('Evidence saved to outputs/EVIDENCE_BASED_FINDINGS.txt')

## Final integration — NewsBotIntelligenceSystem
One model pipeline is reused at inference; we **never** attach dummy numerical features of the wrong shape. For SVM output, raw decision margins are not probabilities. A softmax-transformed margin is a **relative display score**, explicitly uncalibrated, and should not be read as 85% real-world certainty. The system returns labels, POS, approximate SVO, NER, sentiment and lexical emotion proxy in one call.

In [ ]:
class NewsBotIntelligenceSystem:
    def __init__(self,model,nlp_engine,sentiment_engine):
        self.model=model;self.nlp=nlp_engine;self.sentiment_engine=sentiment_engine
    def process_article(self,title,content):
        if not isinstance(title,str) or not isinstance(content,str) or not (title+content).strip():
            raise ValueError('Provide nonempty text title/content.')
        raw=normalize_text(title+' '+content)
        clean=normalize_text(raw)
        pred=str(self.model.predict([clean])[0]);classifier=self.model.named_steps['classifier']
        if hasattr(self.model,'predict_proba'):
            try: scores=np.asarray(self.model.predict_proba([clean])[0],dtype=float)
            except AttributeError: scores=None
        else:scores=None
        if scores is not None:
            score_kind='model_estimated_probability_not_calibrated';display_scores=scores
        else:
            margins=np.atleast_1d(self.model.decision_function([clean])[0]).astype(float)
            display_scores=softmax(margins)
            score_kind='softmax_of_margin_UNCALIBRATED_not_probability'
        score_map={str(k):round(float(v),4) for k,v in zip(classifier.classes_,display_scores)}
        doc=self.nlp(raw[:LINGUISTIC_CHAR_LIMIT])
        syntax=extract_syntactic_features(doc)
        return {'title':title,'predicted_category':pred,'score_type':score_kind,
                'relative_category_scores':score_map,'sentiment':analyze_sentiment(raw),
                'emotion_proxy':emotion_proxy(doc),'entities':extract_entities(doc),
                'pos_proportions':analyze_pos_patterns(doc),
                'syntax':{'n_sentences':syntax['n_sentences'],'n_noun_chunks':syntax['n_noun_chunks'],
                          'example_svo_triples':syntax['svos']},
                'review_recommended_heuristic':bool(max(display_scores)<.60),
                'notes':['Article is outside known historical domain unless separately validated.',
                         'Review cutoff of 0.60 is a demonstration-only heuristic, not validated.',
                         'Score is not a guaranteed reliability measure.',
                         'Emotion and SVO are heuristic annotations.']}

newsbot=NewsBotIntelligenceSystem(best_model,nlp,sia)
print('Integrated engine ready',type(best_model.named_steps['classifier']).__name__)

In [ ]:
# Artificial NEW examples deliberately written for the demonstration, not asserted to be real events.
new_articles=[
 {'title':'Fictional technology trial in Cambridge',
 'content':'Researchers at Cambridge Robotics Laboratory described a new computer chip for artificial intelligence. '
 'The engineers said the prototype accelerates machine learning calculations and consumes less electricity. '
 'The fictional project remains in testing and has not been released to consumers.'},
 {'title':'Fictional regional football cup final',
 'content':'A local football club defeated its rival by two goals in a tournament final on Sunday. '
 'The striker scored late in the match, while the goalkeeper made several saves. '
 'Supporters celebrated the sporting victory after the referee ended the game.'},
 {'title':'Fictional parliamentary budget discussion',
 'content':'Members of parliament debated proposed tax rules during a legislative session. '
 'The prime minister presented a draft policy, and opposition lawmakers demanded changes. '
 'A parliamentary committee will review the proposal next week.'}
]
for a in new_articles:
    result=newsbot.process_article(a['title'],a['content'])
    print('\n',a['title'],'→',result['predicted_category'],'|',result['score_type'],
          '| sentiment:',result['sentiment']['sentiment_label'],'| entities:',result['entities'][:3])
assert len(new_articles)>=2
try:newsbot.process_article('','')
except ValueError:print('Correctly rejected empty input')
else:raise AssertionError('Empty text should be rejected')
print('Three live demonstrations executed; these are synthetic unseen examples, not ground-truth accuracy tests.')

In [ ]:
# Save lightweight project artifacts and reproducibility metadata.
# Never pickle and deploy models from unknown/untrusted sources.
model_path=OUT/'newsbot_classifier.joblib'
dump(best_model,model_path)
manifest={'seed':SEED,'rows':len(df),'train_rows':len(X_train),'holdout_rows':len(X_test),
          'selected_model':best_model_name,'selected_parameters':search.best_params_,
          'cv_macro_f1':float(search.best_score_),
          'holdout_accuracy':metrics['accuracy'],'holdout_macro_f1':metrics['macro_f1'],
          'python':platform.python_version(),'sklearn':__import__('sklearn').__version__,
          'spaCy':spacy.__version__,'linguistic_char_cap':LINGUISTIC_CHAR_LIMIT,
          'source_file_sha256':hashlib.sha256(Path(found).read_bytes()).hexdigest(),
          'dataset_origin':DATA_ORIGIN, 'source_url':'https://www.kaggle.com/competitions/learn-ai-bbc/data',
          'public_research_mirror_url':public_mirror if DATA_ORIGIN.startswith('public_research') else None,
          'license_note':'BBC rights reserved; do not publish raw news text',
          'notebook_execution_status':'REAL_DATA_EXECUTED' if len(df)>=500 else 'INVALID'}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved model, metrics, figures and reproducibility manifest to:',OUT.resolve())
print('Verify all files in the Colab file browser and download outputs before session ends.')

In [ ]:
# End-to-end release checks: FAIL loudly on missing required output files.
expected_figures=['01_data_distribution.png','02_tfidf_category_terms.png',
                  '03_linguistic_sentiment.png','04_ner_analysis.png',
                  '05_model_cv_comparison.png','06_confusion_matrix.png',
                  '07_entity_comentions.png','08_robustness_stress_test.png']
expected_tables=['robustness_stress_test.csv','bootstrap_uncertainty.json','feature_ablation.csv','model_cv_comparison.csv','per_class_metrics.csv','error_analysis_excerpt.csv',
                 'classification_report.txt','holdout_metrics.json','insights.json',
                 'run_manifest.json','newsbot_classifier.joblib','EVIDENCE_BASED_FINDINGS.txt']
missing_outputs=[f for f in (expected_figures+expected_tables) if not (OUT/f).is_file()]
assert not missing_outputs, 'Release incomplete: '+str(missing_outputs)
assert (OUT/'run_manifest.json').exists()
assert 0<=metrics['macro_f1']<=1
assert len(per_class)==df.category.nunique()
assert len(y_train)+len(y_test)==len(df)
assert df.category.nunique()==5
print('PASS: notebook produced 8 figures, required CSV/JSON outputs, model and runtime findings.')
print('NOT VERIFIED BY THIS CELL: professor authorization, data-source rights, external temporal performance, '
      'team contributions, GitHub links, 2-page reflection and 5–7 minute optional video.')

## Research limitations and interpretation checklist

1. **Selection:** Training-only five-fold CV determines algorithm and hyperparameters; the stratified holdout is untouched until the final estimate.
2. **Descriptive vs predictive:** Corpus-wide TF-IDF and POS/NER summaries are exploratory and have not been used as classifier features or for tuning.
3. **Out-of-period risk:** The 2004–2005 BBC articles cannot establish performance on other publishers, 2026 news, or health topics.
4. **Data provenance:** `run_manifest.json` records the actual local input SHA-256 and retrieval origin. Keep any third-party article texts out of the public GitHub repository.
5. **Business value:** Treat classification as editorial routing suggestions, not automated truth verification. Manually inspect typical confusion pairs and fragile entity recognition.


## Conclusions and final review

The completed experiment should be interpreted alongside its original dataset, evaluation design and limitations. The stratified holdout estimates performance on historical BBC articles, not on present-day reporting from unfamiliar publishers. TF-IDF terms, grammatical patterns, sentiment and entity counts offer complementary ways to inspect the same news corpus.

**Practical application:** An editor could review suggested article categories, especially ambiguous stories, and use the entity and linguistic summaries as supporting information. Before operational use, the system would need external validation, human-review rules and monitoring for changing news vocabulary.

**Before submitting:** Confirm the instructor's dataset and assignment-format requirements, review the reflective journal against firsthand experience, and verify that the repository, two PDFs and optional video link (if recorded) can be opened publicly.

**Project repository:** https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System